In [ ]:
# ========== IMPORTS (your existing ones) ==========

import os
import shutil   # for deleting folders

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ========== HELPER FUNCTIONS FOR BALANCING ==========
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff'}

def get_image_files(folder_path):
    """Return list of full paths to image files in folder."""
    image_files = []
    if not os.path.isdir(folder_path):
        return image_files
    for f in os.listdir(folder_path):
        full_path = os.path.join(folder_path, f)
        if os.path.isfile(full_path) and os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
            image_files.append(full_path)
    return image_files

def delete_folder(folder_path):
    """Permanently delete a folder and all its contents."""
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Deleted folder: {folder_path}")

def balance_folder(folder_path, keep_count):
    """Randomly delete images from folder until only `keep_count` remain."""
    images = get_image_files(folder_path)
    if len(images) <= keep_count:
        return 0
    to_delete = random.sample(images, len(images) - keep_count)
    for img_path in to_delete:
        os.remove(img_path)
    print(f"  {os.path.basename(folder_path)}: removed {len(to_delete)} images, now {keep_count}")
    return len(to_delete)

def balance_dataset_inplace(directory, folder_to_remove="disgust"):
    """
    For a given directory (e.g., train/ or test/):
      1. Delete the folder named `folder_to_remove` if it exists.
      2. Balance the remaining class folders to the minimum image count.
    """
    if not os.path.isdir(directory):
        print(f"Directory not found: {directory}")
        return

    # Step 1: Delete the specified folder (e.g., disgust)
    remove_path = os.path.join(directory, folder_to_remove)
    if os.path.exists(remove_path):
        delete_folder(remove_path)
    else:
        print(f"Folder '{folder_to_remove}' not found in {directory} – skipping deletion.")

    # Step 2: Get remaining class folders
    class_folders = [os.path.join(directory, d) for d in os.listdir(directory)
                     if os.path.isdir(os.path.join(directory, d))]
    if not class_folders:
        print(f"No class folders left in {directory} after deletion.")
        return

    # Step 3: Count images in each folder
    counts = {}
    for folder in class_folders:
        counts[folder] = len(get_image_files(folder))
        print(f"  {os.path.basename(folder)}: {counts[folder]} images")

    # Step 4: Minimum count
    min_count = min(counts.values())
    print(f"Minimum count in {directory}: {min_count}")

    # Step 5: Balance each folder
    for folder, cnt in counts.items():
        if cnt > min_count:
            balance_folder(folder, min_count)
        else:
            print(f"  {os.path.basename(folder)} already has {cnt} (≤ {min_count}) – no change")

# ========== APPLY BALANCING TO TRAIN AND TEST ==========
TRAIN_DIR = "/content/drive/My Drive/emotions/train/"
TEST_DIR = "/content/drive/My Drive/emotions/test/"

print("Balancing TRAIN set...")
balance_dataset_inplace(TRAIN_DIR, folder_to_remove="disgust")

print("\nBalancing TEST set...")
balance_dataset_inplace(TEST_DIR, folder_to_remove="disgust")

# ========== YOUR ORIGINAL load_data FUNCTION (unchanged) ==========
def load_data(directory):
    image_paths = []
    labels = []
    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        if not os.path.isdir(label_dir):
            continue
        for filename in os.listdir(label_dir):
            image_path = os.path.join(label_dir, filename)
            image_paths.append(image_path)
            labels.append(label)
        print(label, "done")
    return image_paths, labels

# ========== LOAD THE (NOW BALANCED) DATA ==========
train_paths, train_labels = load_data(TRAIN_DIR)
test_paths, test_labels = load_data(TEST_DIR)

print(f"\nTrain samples: {len(train_paths)}")
print(f"Test samples: {len(test_paths)}")

# ========== (OPTIONAL) VERIFY CLASS DISTRIBUTION ==========
from collections import Counter
print("\nTrain class distribution after balancing:")
print(Counter(train_labels))
print("\nTest class distribution after balancing:")
print(Counter(test_labels))

# ========== CONTINUE WITH YOUR MODEL TRAINING ==========
# ... (your code for loading images into arrays, preprocessing, building model, etc.)

Mounted at /content/drive
Balancing TRAIN set...
Deleted folder: /content/drive/My Drive/emotions/train/disgust
  neutral: 4965 images
  fear: 4097 images
  sad: 4830 images
  surprise: 3171 images
  happy: 7215 images
  angry: 3995 images
Minimum count in /content/drive/My Drive/emotions/train/: 3171
  neutral: removed 1794 images, now 3171
  fear: removed 926 images, now 3171
  sad: removed 1659 images, now 3171
  surprise already has 3171 (≤ 3171) – no change
  happy: removed 4044 images, now 3171
  angry: removed 824 images, now 3171

Balancing TEST set...
Deleted folder: /content/drive/My Drive/emotions/test/disgust
  angry: 958 images
  fear: 1024 images
  happy: 1774 images
  sad: 1247 images
  surprise: 831 images
  neutral: 1233 images
Minimum count in /content/drive/My Drive/emotions/test/: 831
  angry: removed 127 images, now 831
  fear: removed 193 images, now 831
  happy: removed 943 images, now 831
  sad: removed 416 images, now 831
  surprise already has 831 (≤ 831) – no

In [ ]:

# Original paths (already balanced in place)
TRAIN_DIR = "/content/drive/My Drive/emotions/train/"
TEST_DIR = "/content/drive/My Drive/emotions/test/"

# New backup folder
BACKUP_ROOT = "/content/drive/My Drive/emotions_balanced/"
if not os.path.exists(BACKUP_ROOT):
    os.makedirs(BACKUP_ROOT)

# Copy the balanced train and test folders
shutil.copytree(TRAIN_DIR, os.path.join(BACKUP_ROOT, "train"), dirs_exist_ok=True)
shutil.copytree(TEST_DIR, os.path.join(BACKUP_ROOT, "test"), dirs_exist_ok=True)

print(f"Balanced dataset copied to {BACKUP_ROOT}")

Balanced dataset copied to /content/drive/My Drive/emotions_balanced/


In [ ]:
def count_images_in_folder(root_dir, recursive=True, image_extensions=None):
    """
    Count image files in a directory.

    Parameters:
        root_dir (str): Path to the folder to analyze.
        recursive (bool): If True, count images in subfolders as well.
                          If False, only count directly inside root_dir.
        image_extensions (set): Set of allowed extensions (e.g., {'.jpg', '.png'}).
                                If None, uses default common image extensions.

    Returns:
        dict: A dictionary with:
              - 'total': total number of images found
              - 'per_folder': dict mapping subfolder paths to counts (if recursive)
              - 'files': list of image file paths (optional, set via return_files=False)
    """
    if image_extensions is None:
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff'}

    image_extensions = {ext.lower() for ext in image_extensions}
    result = {'total': 0, 'per_folder': {}}

    if recursive:
        # Walk through all subdirectories
        for dirpath, dirnames, filenames in os.walk(root_dir):
            count = 0
            for f in filenames:
                if os.path.splitext(f)[1].lower() in image_extensions:
                    count += 1
            if count > 0:
                # Store count relative to root_dir for readability
                rel_path = os.path.relpath(dirpath, root_dir)
                if rel_path == '.':
                    rel_path = 'root'
                result['per_folder'][rel_path] = count
                result['total'] += count
    else:
        # Only count files directly in root_dir (no subfolders)
        count = 0
        for f in os.listdir(root_dir):
            full_path = os.path.join(root_dir, f)
            if os.path.isfile(full_path) and os.path.splitext(f)[1].lower() in image_extensions:
                count += 1
        result['per_folder'][root_dir] = count
        result['total'] = count

    return result

# Example usage after balancing and saving to new folder:
balanced_path = "/content/drive/My Drive/emotions_balanced/"

print("Counting images in balanced dataset...")
counts = count_images_in_folder(balanced_path, recursive=True)

print(f"\nTotal images: {counts['total']}")
print("\nPer class (subfolder) counts:")
for folder, cnt in counts['per_folder'].items():
    print(f"  {folder}: {cnt} images")

Counting images in balanced dataset...

Total images: 24012

Per class (subfolder) counts:
  train/neutral: 3171 images
  train/fear: 3171 images
  train/sad: 3171 images
  train/surprise: 3171 images
  train/happy: 3171 images
  train/angry: 3171 images
  test/angry: 831 images
  test/fear: 831 images
  test/happy: 831 images
  test/sad: 831 images
  test/surprise: 831 images
  test/neutral: 831 images
